<a href="https://colab.research.google.com/github/Qjh123123/pawplan/blob/main/run_spark_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Java, Spark, and Findspark
This installs Apache Spark 2.2.1, Java 8, and [Findspark](https://github.com/minrk/findspark), a library that makes it easy for Python to find Spark.

In [5]:
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless
%pip install -q "pyspark[connect]==4.0.0"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# Set Environment Variables
Set the locations where Spark and Java are installed.

In [6]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-2.2.1-bin-hadoop2.7"

# Start a SparkSession
This will start a local Spark session.

In [9]:
import os
import sys
from pathlib import Path
import pyspark

# 使用 pip 安装的 Spark，覆盖旧教程的路径
os.environ["SPARK_HOME"] = str(Path(pyspark.__file__).parent)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DMT413")
    .getOrCreate()
)

print("Spark version:", spark.version)
spark.range(5).show()

Spark version: 4.0.0
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



# Use Spark!
That's all there is to it - you're ready to use Spark!

In [10]:
df = spark.createDataFrame([{"hello": "world"} for x in range(1000)])
df.show(3)

+-----+
|hello|
+-----+
|world|
|world|
|world|
+-----+
only showing top 3 rows


In [18]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

event_rows = [
    ("T1", 4), ("T2", 6), ("T1", 7), ("T3", 3),
    ("T2", 5), ("T3", 8), ("T1", 2), ("T4", 4)
]

event_schema = StructType([
    StructField("Route", StringType(), False),
    StructField("Passengers", IntegerType(), False)
])

print(type(event_schema))

events = spark.createDataFrame(event_rows, event_schema)
events.createOrReplaceTempView("events")
events.show()

<class 'pyspark.sql.types.StructType'>
+-----+----------+
|Route|Passengers|
+-----+----------+
|   T1|         4|
|   T2|         6|
|   T1|         7|
|   T3|         3|
|   T2|         5|
|   T3|         8|
|   T1|         2|
|   T4|         4|
+-----+----------+



In [19]:
# Q1(b) and (c)
spark.sql("""
    SELECT Route,
           SUM(Passengers) AS TotalPassengers
    FROM events
    GROUP BY Route
    ORDER BY TotalPassengers DESC, Route ASC
""").show()

+-----+---------------+
|Route|TotalPassengers|
+-----+---------------+
|   T1|             13|
|   T2|             11|
|   T3|             11|
|   T4|              4|
+-----+---------------+



In [20]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

transaction_rows = [
    ("P1", "Electronics", 640.0),
    ("P2", "Clothing",    180.0),
    ("P1", "Electronics", 360.0),
    ("P3", "Electronics", 950.0),
    ("P2", "Electronics", 420.0),
    ("P3", "Clothing",    250.0),
    ("P4", "Electronics", 950.0),
    ("P4", "Clothing",    300.0)
]

transaction_schema = StructType([
    StructField("Product", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("Revenue", DoubleType(), False)
])

transactions = spark.createDataFrame(
    transaction_rows, transaction_schema
)
transactions.createOrReplaceTempView("transactions")

spark.sql("""
    SELECT Product, Revenue
    FROM transactions
    WHERE Category = 'Electronics'
""").show()

+-------+-------+
|Product|Revenue|
+-------+-------+
|     P1|  640.0|
|     P1|  360.0|
|     P3|  950.0|
|     P2|  420.0|
|     P4|  950.0|
+-------+-------+



In [21]:
spark.sql("""
    SELECT Product,
           SUM(Revenue) AS TotalRevenue
    FROM transactions
    WHERE Category = 'Electronics'
    GROUP BY Product
    ORDER BY TotalRevenue DESC, Product ASC
""").show()

+-------+------------+
|Product|TotalRevenue|
+-------+------------+
|     P1|      1000.0|
|     P3|       950.0|
|     P4|       950.0|
|     P2|       420.0|
+-------+------------+



In [22]:
spark.sql("""
    SELECT COUNT(*) AS TransactionCount,
           AVG(Revenue) AS AverageRevenue,
           MAX(Revenue) AS MaximumRevenue
    FROM transactions
    WHERE Category = 'Electronics'
""").show()

+----------------+--------------+--------------+
|TransactionCount|AverageRevenue|MaximumRevenue|
+----------------+--------------+--------------+
|               5|         664.0|         950.0|
+----------------+--------------+--------------+



In [23]:
q2_result = (
    transactions
    .filter(F.col("Category") == "Electronics")
    .groupBy("Product")
    .agg(F.sum("Revenue").alias("TotalRevenue"))
    .orderBy(
        F.col("TotalRevenue").desc(),
        F.col("Product").asc()
    )
)

q2_result.show()

+-------+------------+
|Product|TotalRevenue|
+-------+------------+
|     P1|      1000.0|
|     P3|       950.0|
|     P4|       950.0|
|     P2|       420.0|
+-------+------------+



In [24]:
people_rows = [
    ("Amir", 32),
    ("Bella", 46),
    ("Chen", 46),
    ("Devi", 28),
    ("Evan", 39),
    ("Fara", 35)
]

people_schema = StructType([
    StructField("Name", StringType(), False),
    StructField("Age", IntegerType(), False)
])

department_rows = [
    ("Amir", "Operations"),
    ("Bella", "Finance"),
    ("Chen", "IT"),
    ("Devi", "HR"),
    ("Evan", "Operations"),
    ("Fara", "IT")
]

department_schema = StructType([
    StructField("Name", StringType(), False),
    StructField("Department", StringType(), False)
])

people = spark.createDataFrame(people_rows, people_schema)
departments = spark.createDataFrame(
    department_rows, department_schema
)

people.createOrReplaceTempView("people")
departments.createOrReplaceTempView("departments")

people.show()
departments.show()

+-----+---+
| Name|Age|
+-----+---+
| Amir| 32|
|Bella| 46|
| Chen| 46|
| Devi| 28|
| Evan| 39|
| Fara| 35|
+-----+---+

+-----+----------+
| Name|Department|
+-----+----------+
| Amir|Operations|
|Bella|   Finance|
| Chen|        IT|
| Devi|        HR|
| Evan|Operations|
| Fara|        IT|
+-----+----------+



In [25]:
spark.sql("""
    SELECT p.Name,
           p.Age,
           d.Department
    FROM people AS p
    INNER JOIN departments AS d
        ON p.Name = d.Name
    ORDER BY p.Age DESC, p.Name ASC
""").show()

+-----+---+----------+
| Name|Age|Department|
+-----+---+----------+
|Bella| 46|   Finance|
| Chen| 46|        IT|
| Evan| 39|Operations|
| Fara| 35|        IT|
| Amir| 32|Operations|
| Devi| 28|        HR|
+-----+---+----------+



In [26]:
spark.sql("""
    SELECT p.Name,
           p.Age,
           d.Department
    FROM people AS p
    INNER JOIN departments AS d
        ON p.Name = d.Name
    WHERE p.Age = (
        SELECT MAX(Age)
        FROM people
    )
    ORDER BY p.Name ASC
""").show()

+-----+---+----------+
| Name|Age|Department|
+-----+---+----------+
|Bella| 46|   Finance|
| Chen| 46|        IT|
+-----+---+----------+

